In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
from tkcalendar import DateEntry
import sqlite3
from datetime import date

class LibraryApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Library Management System")
        self.root.geometry("1000x600")
        
        main_frame = ttk.Frame(root, padding="10")
        main_frame.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        form_frame = ttk.LabelFrame(main_frame, text="Borrow Book Form", padding="10")
        form_frame.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S), padx=(0, 10))
        
        ttk.Label(form_frame, text="Student Name:").grid(row=0, column=0, sticky=tk.W, pady=5)
        self.student_name = ttk.Entry(form_frame, width=25)
        self.student_name.grid(row=0, column=1, sticky=(tk.W, tk.E), pady=5, padx=5)
        
        ttk.Label(form_frame, text="Select Book:").grid(row=1, column=0, sticky=tk.W, pady=5)
        self.book_combo = ttk.Combobox(form_frame, width=23, state="readonly")
        self.book_combo.grid(row=1, column=1, sticky=(tk.W, tk.E), pady=5, padx=5)
        
        ttk.Label(form_frame, text="Borrow Date:").grid(row=2, column=0, sticky=tk.W, pady=5)
        self.borrow_date = DateEntry(form_frame, width=22, date_pattern='y-mm-dd')
        self.borrow_date.grid(row=2, column=1, sticky=tk.W, pady=5, padx=5)
        
        ttk.Label(form_frame, text="Return Date:").grid(row=3, column=0, sticky=tk.W, pady=5)
        self.return_date = DateEntry(form_frame, width=22, date_pattern='y-mm-dd')
        self.return_date.grid(row=3, column=1, sticky=tk.W, pady=5, padx=5)
        
        button_frame = ttk.Frame(form_frame)
        button_frame.grid(row=4, column=0, columnspan=2, pady=15)
        tk.Button(button_frame, text="Add", command=self.borrow_book, bg="lightgreen").grid(row=0, column=0, padx=3)
        tk.Button(button_frame, text="Delete", command=self.delete_record, bg="lightcoral").grid(row=0, column=1, padx=3)
        tk.Button(button_frame, text="Clear", command=self.clear_form, bg="lightgray").grid(row=0, column=2, padx=3)
    
        table_frame = ttk.LabelFrame(main_frame, text="Borrowed Books", padding="10")
        table_frame.grid(row=0, column=1, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        search_frame = ttk.Frame(table_frame)
        search_frame.grid(row=0, column=0, sticky=(tk.W, tk.E), pady=(0, 10))
        ttk.Label(search_frame, text="Search:").grid(row=0, column=0, sticky=tk.W, padx=(0, 5))
        self.search_var = tk.StringVar()
        ttk.Entry(search_frame, textvariable=self.search_var, width=30).grid(row=0, column=1, sticky=(tk.W, tk.E), padx=5)
        self.search_var.trace('w', self.filter_records)
        self.record_count = ttk.Label(search_frame, text="Records: 0")
        self.record_count.grid(row=0, column=2, sticky=tk.E, padx=(10, 0))
        
        tree_frame = ttk.Frame(table_frame)
        tree_frame.grid(row=1, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        columns = ("ID", "Student", "Book", "Borrow Date", "Return Date", "Fine")
        self.tree = ttk.Treeview(tree_frame, columns=columns, show="headings", height=20)
        
        style = ttk.Style()
        style.theme_use('default')
        style.configure("Treeview.Heading", background="blue", foreground="white", font=('Arial', 10, 'bold'))
        
        for col in columns: self.tree.heading(col, text=col)
        self.tree.column("ID", width=60, anchor=tk.CENTER)
        self.tree.column("Student", width=120)
        self.tree.column("Book", width=180)
        self.tree.column("Borrow Date", width=100)
        self.tree.column("Return Date", width=100)
        self.tree.column("Fine", width=80, anchor=tk.CENTER)
        self.tree.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        self.tree.tag_configure('fine_owed', background='lightcoral')
        
        scrollbar = ttk.Scrollbar(tree_frame, orient=tk.VERTICAL, command=self.tree.yview)
        scrollbar.grid(row=0, column=1, sticky=(tk.N, tk.S))
        self.tree.configure(yscrollcommand=scrollbar.set)
        
        root.columnconfigure(0, weight=1); root.rowconfigure(0, weight=1)
        main_frame.columnconfigure(0, weight=1); main_frame.columnconfigure(1, weight=3); main_frame.rowconfigure(0, weight=1)
        form_frame.columnconfigure(1, weight=1); table_frame.columnconfigure(0, weight=1); table_frame.rowconfigure(1, weight=1)
        search_frame.columnconfigure(1, weight=1); tree_frame.columnconfigure(0, weight=1); tree_frame.rowconfigure(0, weight=1)
        
        today = date.today()
        self.borrow_date.set_date(today); self.return_date.set_date(today)
        self.borrow_date.bind("<<DateEntrySelected>>", self.validate_dates)
        self.return_date.bind("<<DateEntrySelected>>", self.validate_dates)
        self.load_available_books(); self.load_borrowed_books()
    
    def load_available_books(self):
        conn = sqlite3.connect("library.db")
        cursor = conn.cursor()
        cursor.execute("SELECT id, title FROM books WHERE quantity > 0")
        self.book_combo['values'] = [f"{book[1]} (ID: {book[0]})" for book in cursor.fetchall()]
        conn.close()
    
    def filter_records(self, *args):
        search_term = self.search_var.get().lower()
        for item in self.tree.get_children(): self.tree.delete(item)
        conn = sqlite3.connect("library.db")
        cursor = conn.cursor()
        cursor.execute('SELECT bb.id, bb.student_name, b.title, bb.borrow_date, bb.return_date, bb.fine FROM borrowed_books bb JOIN books b ON bb.book_id = b.id')
        record_count = 0
        for book in cursor.fetchall():
            if search_term in book[1].lower() or search_term in book[2].lower():
                item_id = self.tree.insert("", tk.END, values=book)
                record_count += 1
                if book[5] > 0: self.tree.item(item_id, tags=('fine_owed',))
        self.record_count.config(text=f"Records: {record_count}")
        conn.close()
    
    def load_borrowed_books(self):
        self.search_var.set(""); self.filter_records()
    
    def validate_dates(self, event=None):
        borrow_date = self.borrow_date.get_date(); return_date = self.return_date.get_date(); today = date.today()
        if borrow_date < today: messagebox.showerror("Invalid Date", "Borrow date cannot be in the past!"); self.borrow_date.set_date(today); return False
        if return_date < borrow_date: messagebox.showerror("Invalid Date", "Return date cannot be before borrow date!"); self.return_date.set_date(borrow_date); return False
        return True
    
    def calculate_fine(self, return_date_obj):
        today = date.today()
        return (today - return_date_obj).days * 5 if return_date_obj < today else 0
    
    def borrow_book(self):
        if not self.validate_dates(): return
        student_name = self.student_name.get().strip(); book_selection = self.book_combo.get()
        if not student_name: messagebox.showerror("Input Error", "Please enter student name!"); return
        if not book_selection: messagebox.showerror("Input Error", "Please select a book!"); return
        try: book_id = int(book_selection.split("(ID: ")[1].rstrip(")"))
        except: messagebox.showerror("Error", "Invalid book selection!"); return
        borrow_date_obj = self.borrow_date.get_date(); return_date_obj = self.return_date.get_date()
        fine = self.calculate_fine(return_date_obj)
        try:
            conn = sqlite3.connect("library.db")
            cursor = conn.cursor()
            cursor.execute('INSERT INTO borrowed_books (student_name, book_id, borrow_date, return_date, fine) VALUES (?, ?, ?, ?, ?)', 
                         (student_name, book_id, borrow_date_obj.strftime('%Y-%m-%d'), return_date_obj.strftime('%Y-%m-%d'), fine))
            cursor.execute("UPDATE books SET quantity = quantity - 1 WHERE id = ?", (book_id,))
            conn.commit(); conn.close()
            messagebox.showinfo("Success", "Book borrowed successfully!")
            self.clear_form(); self.load_available_books(); self.load_borrowed_books()
        except sqlite3.Error as e: messagebox.showerror("Database Error", f"Error: {e}")
    
    def delete_record(self):
        selected_item = self.tree.selection()
        if not selected_item: messagebox.showwarning("Selection Error", "Please select a record to delete!"); return
        if not messagebox.askyesno("Confirm Delete", "Are you sure?"): return
        record_id = self.tree.item(selected_item[0])['values'][0]
        try:
            conn = sqlite3.connect("library.db")
            cursor = conn.cursor()
            cursor.execute("SELECT book_id FROM borrowed_books WHERE id = ?", (record_id,))
            book_id = cursor.fetchone()[0]
            cursor.execute("DELETE FROM borrowed_books WHERE id = ?", (record_id,))
            cursor.execute("UPDATE books SET quantity = quantity + 1 WHERE id = ?", (book_id,))
            conn.commit(); conn.close()
            messagebox.showinfo("Success", "Record deleted successfully!")
            self.load_available_books(); self.load_borrowed_books()
        except sqlite3.Error as e: messagebox.showerror("Database Error", f"Error: {e}")
    
    def clear_form(self):
        self.student_name.delete(0, tk.END); self.book_combo.set('')
        today = date.today(); self.borrow_date.set_date(today); self.return_date.set_date(today)

if __name__ == "__main__":
    root = tk.Tk()
    app = LibraryApp(root)
    root.mainloop()